# SSAFY AI Challenge - Improved VQA Baseline

## 개선 사항
- **Step 1**: 모델 변경 → `openbmb/MiniCPM-V-2` (2.8B Vision-Language Model)
- **Step 2**: 전체 train 데이터 사용 (200개 샘플링 제거)
- **Step 3**: dev 데이터를 활용한 데이터 증강 (응답1~5 majority voting으로 pseudo-label 생성)
- **Step 4**: 프롬프트 개선 (한국어 특화, 구조화된 지시사항)
- **Step 5**: Epoch 증가 (1 → 3)
- **Step 6**: 하이퍼파라미터 조정 (lr, LoRA rank, scheduler 등)
- **Step 7**: YOLO 객체 탐지 결과를 VLM 프롬프트에 주입 (Knowledge Distillation)

실행 환경: **Google Colab Pro (A100 40GB)**

# 0. 환경 준비

In [ ]:
# 필수 라이브러리 설치
# MiniCPM-V-2는 trust_remote_code 기반 커스텀 모델이므로 transformers 최신 버전 필요
!pip -q install transformers>=4.40.0 accelerate
!pip -q install "peft>=0.13.2" "bitsandbytes>=0.43.0" datasets pillow pandas tqdm
!pip -q install ultralytics  # Step 7: YOLOv8 객체 탐지

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# 1. 데이터 준비 (Google Drive 마운트 및 압축 해제)

In [ ]:
import os

# 데이터가 세션에 직접 업로드된 경우 경로 확인
# Colab 좌측 파일 탭에서 실제 경로를 확인 후 DATA_DIR을 수정하세요
DATA_DIR = '/content'

# 업로드된 파일 목록 확인
for f in ['train.csv', 'test.csv', 'dev.csv', 'sample_submission.csv']:
    path = os.path.join(DATA_DIR, f)
    exists = os.path.exists(path)
    print(f'{f}: {"OK" if exists else "NOT FOUND"} ({path})')

# train/test 폴더 확인
for d in ['train', 'test', 'dev']:
    path = os.path.join(DATA_DIR, d)
    if os.path.isdir(path):
        print(f'{d}/: OK ({len(os.listdir(path))} files)')
    else:
        print(f'{d}/: NOT FOUND')

In [ ]:
import os, glob

# 세션에 업로드된 zip 파일 자동 탐색 후 압축 해제
zip_files = glob.glob('/content/*.zip')

if zip_files:
    zip_path = zip_files[0]
    print(f'zip 파일 발견: {zip_path}')
    !unzip -q "{zip_path}" -d "/content/"
    print('압축 해제 완료')
else:
    print('zip 파일 없음 - 데이터가 이미 압축 해제되어 있거나 직접 업로드된 상태입니다')

# 압축 해제 후 파일 구조 확인
print('\n/content 내 파일 목록:')
for item in sorted(os.listdir('/content')):
    print(' ', item)

# 2. 라이브러리 임포트 및 전역 설정

In [ ]:
import os, re, math, random, json
from collections import Counter
from dataclasses import dataclass
from typing import Any, List, Dict, Optional

import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

Image.MAX_IMAGE_PIXELS = None

# ── 전역 설정 ──────────────────────────────────────────────────
SEED        = 42
MODEL_ID    = "openbmb/MiniCPM-V-2"   # Step 1: 모델 변경
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# 학습 하이퍼파라미터 (Step 5, 6)
NUM_EPOCHS      = 3          # Step 5: 1→3
BATCH_SIZE      = 2          # A100 40GB에서 배치 증가
GRAD_ACCUM      = 4
LR              = 5e-5       # Step 6: 1e-4→5e-5 (안정적 수렴)
WARMUP_RATIO    = 0.05
MAX_NEW_TOKENS  = 8
LORA_R          = 16         # Step 6: 8→16 (표현력 증가)
LORA_ALPHA      = 32         # Step 6: 16→32
LORA_DROPOUT    = 0.05
WEIGHT_DECAY    = 0.01       # Step 6: AdamW weight decay 추가
MAX_GRAD_NORM   = 1.0        # Step 6: gradient clipping

# Step 7: YOLO 사용 여부
USE_YOLO        = True
YOLO_CONF       = 0.25       # 탐지 신뢰도 임계값
YOLO_MAX_OBJ    = 5          # 프롬프트에 포함할 최대 객체 수

# 경로
DATA_DIR    = "/content"
SAVE_DIR    = "/content/minicpm_v2_lora"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE} | Model: {MODEL_ID}")
print(f"Epochs: {NUM_EPOCHS} | LR: {LR} | LoRA r: {LORA_R}")

# 3. Step 7 - YOLO 객체 탐지 모듈

YOLOv8n으로 이미지에서 객체를 탐지하고, 탐지 결과를 자연어로 변환하여 VLM 프롬프트에 주입합니다.  
이를 통해 탐지 모델의 spatial knowledge를 VLM에 distill합니다.

In [ ]:
# ── YOLO 탐지기 초기화 ─────────────────────────────────────────
yolo_model = None

if USE_YOLO:
    from ultralytics import YOLO
    # YOLOv8n: 경량 모델, 자동 다운로드
    yolo_model = YOLO("yolov8n.pt")
    print("YOLO 모델 로드 완료")


def detect_objects(img: Image.Image, conf: float = YOLO_CONF, max_obj: int = YOLO_MAX_OBJ) -> str:
    """
    YOLOv8로 이미지에서 객체를 탐지하고
    VLM 프롬프트에 삽입할 자연어 설명을 반환합니다.

    반환 예시:
        "[탐지된 객체: bottle(0.92), cup(0.85), table(0.73)]"
    """
    if yolo_model is None:
        return ""

    try:
        results = yolo_model(img, conf=conf, verbose=False)[0]
        boxes = results.boxes
        if boxes is None or len(boxes) == 0:
            return ""

        # confidence 내림차순 정렬 후 상위 max_obj개
        confs  = boxes.conf.cpu().numpy()
        clsids = boxes.cls.cpu().numpy().astype(int)
        names  = results.names  # {id: label}

        order   = np.argsort(confs)[::-1][:max_obj]
        obj_list = [f"{names[clsids[i]]}({confs[i]:.2f})" for i in order]

        return "[탐지된 객체: " + ", ".join(obj_list) + "]"
    except Exception:
        return ""


# 동작 확인 (선택)
# from PIL import Image as PILImage
# test_img = PILImage.new("RGB", (224, 224), color=(128, 128, 128))
# print(detect_objects(test_img))

# 4. Step 3 - dev 데이터 증강 (Pseudo-labeling)

dev.csv의 `응답1~응답5` 컬럼에 교육생 5명이 제출한 응답이 있습니다.  
Majority Voting으로 과반수 이상(≥3/5) 동의한 항목만 pseudo-label로 사용합니다.

In [ ]:
# ── 데이터 로드 ────────────────────────────────────────────────
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")
dev_df   = pd.read_csv(f"{DATA_DIR}/dev.csv")

print(f"Train: {len(train_df)}개 | Test: {len(test_df)}개 | Dev: {len(dev_df)}개")
print("Dev 컬럼:", dev_df.columns.tolist())

In [ ]:
def make_pseudo_label(row, vote_cols, threshold: int = 3) -> Optional[str]:
    """
    5개 응답 중 threshold개 이상 동일하면 pseudo-label 반환.
    미달이면 None 반환 (불확실한 샘플 제외).
    """
    votes = [str(row[c]).strip().lower() for c in vote_cols if pd.notna(row[c])]
    valid = [v for v in votes if v in ("a", "b", "c", "d")]
    if not valid:
        return None
    most_common, count = Counter(valid).most_common(1)[0]
    return most_common if count >= threshold else None


# 응답 컬럼 탐지 (응답1~응답5)
vote_cols = [c for c in dev_df.columns if c.startswith("응답")]
print(f"응답 컬럼: {vote_cols}")

dev_df["answer"] = dev_df.apply(lambda r: make_pseudo_label(r, vote_cols), axis=1)
dev_pseudo = dev_df.dropna(subset=["answer"]).copy()

print(f"Dev 전체: {len(dev_df)}개 → Pseudo-label 생성: {len(dev_pseudo)}개 "
      f"({len(dev_pseudo)/len(dev_df)*100:.1f}% 신뢰 샘플)")

# 정답 분포 확인
print("Pseudo-label 분포:", dev_pseudo["answer"].value_counts().to_dict())

In [ ]:
# ── Step 2+3: 전체 train + pseudo-labeled dev 합치기 ───────────
# 공통 컬럼만 추출해서 합침
keep_cols = ["id", "path", "question", "a", "b", "c", "d", "answer"]
combined_df = pd.concat(
    [train_df[keep_cols], dev_pseudo[keep_cols]],
    ignore_index=True
).reset_index(drop=True)

print(f"최종 학습 데이터: {len(train_df)}(train) + {len(dev_pseudo)}(dev pseudo) = {len(combined_df)}개")

# 90:10 검증 분리
combined_df = combined_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
split       = int(len(combined_df) * 0.9)
train_subset = combined_df.iloc[:split].reset_index(drop=True)
valid_subset = combined_df.iloc[split:].reset_index(drop=True)

print(f"학습: {len(train_subset)}개 | 검증: {len(valid_subset)}개")

# 5. 모델 & 토크나이저 로드 (MiniCPM-V-2)

In [ ]:
# ── 양자화 설정 (A100에서는 bfloat16으로도 충분하나, VRAM 여유를 위해 4-bit 유지) ─
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # A100은 bfloat16 네이티브 지원
)

print("토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

print("모델 로드 중 (4-bit 양자화)...")
base_model = AutoModel.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

print("모델 로드 완료")

In [ ]:
# ── Step 6: LoRA 설정 강화 (r=16, alpha=32) ────────────────────
# MiniCPM-V-2의 LLM 백본(MiniCPM-2B)에 LoRA 적용
# 적용 가능한 레이어를 먼저 확인
target_modules = []
for name, _ in base_model.named_modules():
    if any(k in name for k in ["q_proj", "k_proj", "v_proj", "o_proj",
                                "gate_proj", "up_proj", "down_proj"]):
        target_modules.append(name.split(".")[-1])

target_modules = list(set(target_modules))
print("LoRA 적용 모듈:", target_modules)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=target_modules if target_modules else ["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 6. Step 4 - 개선된 프롬프트 템플릿

In [ ]:
# ── Step 4: 개선된 프롬프트 ─────────────────────────────────────
# 변경 포인트:
#   1. 한국어 시스템 지시사항으로 도메인 특화
#   2. 선지 포맷 명확화 (선지 레이블 앞뒤 괄호 통일)
#   3. YOLO 탐지 정보 삽입 위치 명시 (Step 7)
#   4. 간결하고 명확한 출력 지시

SYSTEM_INSTRUCT = (
    "당신은 이미지를 분석하여 4지선다 문제를 푸는 전문 AI 어시스턴트입니다. "
    "이미지를 주의 깊게 관찰하고, 질문과 선지를 바탕으로 가장 적합한 답을 선택하세요. "
    "반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요. 설명이나 다른 텍스트는 출력하지 마세요."
)


def build_mc_prompt(question: str, a: str, b: str, c: str, d: str,
                    yolo_info: str = "") -> str:
    """
    Step 4+7: YOLO 탐지 정보가 있으면 질문 앞에 삽입.

    구조:
        [탐지된 객체: ...] (YOLO 있을 때만)
        질문
        (a) 선지A
        (b) 선지B
        (c) 선지C
        (d) 선지D
        정답 (a/b/c/d 중 하나):
    """
    yolo_prefix = f"{yolo_info}\n" if yolo_info else ""
    return (
        f"{yolo_prefix}"
        f"질문: {question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n"
        "정답 (a/b/c/d 중 하나):"
    )


print("프롬프트 예시:")
print(build_mc_prompt(
    "사진 속 물체의 재질은?",
    "유리", "금속", "종이", "플라스틱",
    yolo_info="[탐지된 객체: bottle(0.92), cup(0.85)]"
))

# 7. Dataset & DataLoader

In [ ]:
class VQAMCDataset(Dataset):
    """
    MiniCPM-V-2용 VQA 데이터셋.
    - train=True : 학습용 (정답 포함, teacher forcing)
    - train=False: 추론용 (정답 없음)
    - use_yolo   : YOLO 탐지 정보를 프롬프트에 추가 (Step 7)
    """
    def __init__(self, df: pd.DataFrame, tokenizer, train: bool = True,
                 use_yolo: bool = False):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.train     = train
        self.use_yolo  = use_yolo

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        # Step 7: YOLO 탐지
        yolo_info = detect_objects(img) if self.use_yolo else ""

        user_text = build_mc_prompt(
            str(row["question"]),
            str(row["a"]), str(row["b"]),
            str(row["c"]), str(row["d"]),
            yolo_info=yolo_info,
        )

        # MiniCPM-V-2 conversation 포맷
        msgs = [
            {
                "role": "user",
                "content": user_text,
            }
        ]

        answer = str(row["answer"]).strip().lower() if self.train else None

        return {
            "image"   : img,
            "msgs"    : msgs,
            "answer"  : answer,
            "system"  : SYSTEM_INSTRUCT,
        }


@dataclass
class DataCollator:
    """
    MiniCPM-V-2는 model.chat() 인터페이스를 주로 사용하지만,
    배치 학습을 위해 tokenizer로 직접 인코딩합니다.
    """
    tokenizer: Any
    train: bool = True

    def __call__(self, batch: List[Dict]) -> Dict:
        # MiniCPM-V-2 기본 chat 포맷으로 변환
        images, input_ids_list, labels_list = [], [], []

        for sample in batch:
            img     = sample["image"]
            msgs    = sample["msgs"]
            answer  = sample["answer"]
            system  = sample["system"]

            # 시스템 + 유저 텍스트 조합
            user_content = msgs[0]["content"]
            if answer and self.train:
                # 학습: 전체 시퀀스 (prompt + answer)
                full_text = (
                    f"<|im_start|>system\n{system}<|im_end|>\n"
                    f"<|im_start|>user\n{user_content}<|im_end|>\n"
                    f"<|im_start|>assistant\n{answer}<|im_end|>"
                )
                prompt_text = (
                    f"<|im_start|>system\n{system}<|im_end|>\n"
                    f"<|im_start|>user\n{user_content}<|im_end|>\n"
                    f"<|im_start|>assistant\n"
                )
            else:
                full_text = (
                    f"<|im_start|>system\n{system}<|im_end|>\n"
                    f"<|im_start|>user\n{user_content}<|im_end|>\n"
                    f"<|im_start|>assistant\n"
                )
                prompt_text = full_text

            enc = self.tokenizer(
                full_text,
                return_tensors="pt",
                truncation=True,
                max_length=512,
            )
            input_ids = enc["input_ids"][0]

            if self.train:
                # prompt 부분은 -100 마스킹 (응답 토큰만 loss 계산)
                prompt_enc = self.tokenizer(
                    prompt_text,
                    return_tensors="pt",
                    truncation=True,
                    max_length=512,
                )
                prompt_len = prompt_enc["input_ids"].shape[1]
                labels = input_ids.clone()
                labels[:prompt_len] = -100
                labels_list.append(labels)

            input_ids_list.append(input_ids)
            images.append(img)

        # 패딩
        max_len = max(ids.shape[0] for ids in input_ids_list)
        pad_id  = self.tokenizer.pad_token_id or 0

        padded_ids  = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
        attn_mask   = torch.zeros(len(batch), max_len, dtype=torch.long)

        for j, ids in enumerate(input_ids_list):
            L = ids.shape[0]
            padded_ids[j, :L] = ids
            attn_mask[j, :L]  = 1

        result = {"input_ids": padded_ids, "attention_mask": attn_mask}

        if self.train:
            padded_labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
            for j, lbl in enumerate(labels_list):
                L = lbl.shape[0]
                padded_labels[j, :L] = lbl
            result["labels"] = padded_labels

        result["images"] = images  # 이미지는 리스트로 전달
        return result


# ── DataLoader 생성 ────────────────────────────────────────────
train_ds = VQAMCDataset(train_subset, tokenizer, train=True,  use_yolo=USE_YOLO)
valid_ds = VQAMCDataset(valid_subset, tokenizer, train=True,  use_yolo=USE_YOLO)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=DataCollator(tokenizer, train=True), num_workers=2, pin_memory=True,
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=DataCollator(tokenizer, train=True), num_workers=2, pin_memory=True,
)

print(f"Train batches: {len(train_loader)} | Valid batches: {len(valid_loader)}")

# 8. Step 5+6 - 학습 (3 epochs, 개선된 하이퍼파라미터)

In [ ]:
model = model.to(DEVICE)

# ── Step 6: 옵티마이저 & 스케줄러 개선 ────────────────────────
# AdamW + weight_decay / cosine warmup scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8,
)

total_steps    = NUM_EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
warmup_steps   = int(total_steps * WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(  # cosine 변경 (Step 6)
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=True)

print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")

# ── 학습 루프 ─────────────────────────────────────────────────
best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": []}

for epoch in range(NUM_EPOCHS):
    # ---- Train ----
    model.train()
    running, global_step = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]")

    for step, batch in enumerate(pbar, start=1):
        images = batch.pop("images")  # 이미지는 별도 처리
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}

        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            # Step 6: gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg = running / GRAD_ACCUM
            pbar.set_postfix({"loss": f"{avg:.3f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
            running = 0.0

    epoch_train_loss = avg
    history["train_loss"].append(epoch_train_loss)

    # ---- Validation ----
    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [valid]"):
            vb.pop("images")
            vb = {k: v.to(DEVICE) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1

    avg_val = val_loss / val_steps
    history["val_loss"].append(avg_val)
    print(f"[Epoch {epoch+1}] train_loss={epoch_train_loss:.4f} | val_loss={avg_val:.4f}")

    # Best 모델 저장
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(SAVE_DIR + "_best")
        tokenizer.save_pretrained(SAVE_DIR + "_best")
        print(f"  → Best model saved (val_loss={best_val_loss:.4f})")

    model.train()

# 최종 모델 저장
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("최종 모델 저장 완료:", SAVE_DIR)
print("학습 히스토리:", history)

# 9. 추론 (Inference)

In [ ]:
# ── 응답 파싱 ─────────────────────────────────────────────────
def extract_choice(text: str) -> str:
    """모델 출력에서 a/b/c/d 추출. 못 찾으면 'a' 반환."""
    text = text.strip().lower()

    # 마지막 줄 우선 확인
    for line in reversed(text.splitlines()):
        line = line.strip()
        if line in ("a", "b", "c", "d"):
            return line
        # 괄호 패턴: (a), [a], 'a'
        m = re.search(r"[\(\[\']?([abcd])[\)\]\']?", line)
        if m:
            return m.group(1)

    # 전체 텍스트에서 첫 번째 유효 선지
    m = re.search(r"\b([abcd])\b", text)
    return m.group(1) if m else "a"


# ── Best 모델 로드 ─────────────────────────────────────────────
print("Best 모델 로드 중...")
from peft import PeftModel

infer_model = AutoModel.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
infer_model = PeftModel.from_pretrained(infer_model, SAVE_DIR + "_best")
infer_model.eval()
print("추론 모델 로드 완료")

In [ ]:
# ── 추론 루프 ─────────────────────────────────────────────────
preds = []

for i in tqdm(range(len(test_df)), desc="Inference"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")

    # Step 7: 추론 시에도 YOLO 적용
    yolo_info = detect_objects(img) if USE_YOLO else ""

    user_text = build_mc_prompt(
        str(row["question"]),
        str(row["a"]), str(row["b"]),
        str(row["c"]), str(row["d"]),
        yolo_info=yolo_info,
    )

    # MiniCPM-V-2 model.chat() 인터페이스 활용
    msgs = [{"role": "user", "content": user_text}]

    try:
        response = infer_model.chat(
            image=img,
            msgs=msgs,
            tokenizer=tokenizer,
            system_prompt=SYSTEM_INSTRUCT,
            sampling=False,           # greedy decoding
            max_new_tokens=MAX_NEW_TOKENS,
        )
    except Exception as e:
        # fallback: tokenizer 직접 인코딩 방식
        full_text = (
            f"<|im_start|>system\n{SYSTEM_INSTRUCT}<|im_end|>\n"
            f"<|im_start|>user\n{user_text}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        inputs = tokenizer(full_text, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out_ids = infer_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(out_ids[0], skip_special_tokens=True)

    pred = extract_choice(str(response))
    preds.append(pred)

# ── 제출 파일 생성 ─────────────────────────────────────────────
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)

print("submission.csv 저장 완료")
print("정답 분포:", submission["answer"].value_counts().to_dict())
submission.head()

# 10. (선택) 검증셋 정확도 확인

In [ ]:
# 검증셋으로 accuracy 계산 (학습에 사용하지 않은 데이터)
val_preds, val_golds = [], []

for i in tqdm(range(len(valid_subset)), desc="Val Inference"):
    row = valid_subset.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    yolo_info = detect_objects(img) if USE_YOLO else ""

    user_text = build_mc_prompt(
        str(row["question"]),
        str(row["a"]), str(row["b"]),
        str(row["c"]), str(row["d"]),
        yolo_info=yolo_info,
    )
    msgs = [{"role": "user", "content": user_text}]

    try:
        response = infer_model.chat(
            image=img, msgs=msgs, tokenizer=tokenizer,
            system_prompt=SYSTEM_INSTRUCT,
            sampling=False, max_new_tokens=MAX_NEW_TOKENS,
        )
    except Exception:
        response = "a"

    val_preds.append(extract_choice(str(response)))
    val_golds.append(str(row["answer"]).strip().lower())

acc = sum(p == g for p, g in zip(val_preds, val_golds)) / len(val_golds)
print(f"검증셋 Accuracy: {acc*100:.2f}% ({sum(p==g for p,g in zip(val_preds,val_golds))}/{len(val_golds)})")